In [ ]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from argparse import ArgumentParser
from os import cpu_count

import ipyplot
import torch
import torchvision
from lightning.pytorch import Trainer, cli_lightning_logo
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from torchvision import transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

import notebook_setup  # isort:skip
from acid import conf  # isort:skip
from acid.torch.data import COCOJsonDataset  # isort:skip
from acid.torch.setup import BoardModelSetup  # isort:skip
from acid.torch.lightning_modules import BoardSegmentationModule  # isort:skip
from acid.torch.logging import get_logger  # isort:skip

In [ ]:
EPOCHS = 500
LR = None  # set to None to enable auto_lr_find
IMAGE_SIZE = BoardModelSetup.image_size
NUM_WORKERS = cpu_count() - 2
BATCH_SIZE = 4
USE_WANDB_LOGGING = True

In [ ]:
train_set = COCOJsonDataset(
    BoardModelSetup.dataset_path,
    image_size=BoardModelSetup.image_size,
    split="train",
    transform=BoardModelSetup.transforms["train"],
)
val_set = COCOJsonDataset(
    BoardModelSetup.dataset_path,
    image_size=BoardModelSetup.image_size,
    split="val",
    transform=BoardModelSetup.transforms["val"],
)

In [ ]:
# plot some examples
for idx in range(0, min(len(train_set), 10)):
    image, mask = train_set[idx]
    ipyplot.plot_images([T.ToPILImage()(image)], img_width=300)

In [ ]:
module_params = {}
auto_lr_find = True
if LR is not None:
    auto_lr_find = False
    module_params["lr"] = LR

model = BoardSegmentationModule(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, train_set=train_set, val_set=val_set, **module_params
)

logger = get_logger("Acid Chess - Board Segmentation", conf.NOTEBOOKS_LOGS_DIR, USE_WANDB_LOGGING)
logger.watch(model.net)

checkpoint_callback = ModelCheckpoint(dirpath=conf.NOTEBOOKS_CHECKPOINTS_DIR, save_top_k=10, monitor="val_loss")
trainer = Trainer(
    accelerator="auto",
    default_root_dir=conf.NOTEBOOKS_TMP_DIR,
    logger=logger,
    max_epochs=EPOCHS,
    log_every_n_steps=10,
    callbacks=[checkpoint_callback],
    auto_lr_find=auto_lr_find,
)

In [ ]:
if auto_lr_find is True:
    trainer.tune(model)

trainer.fit(model)